# **CA5** - Predicting Cognitive Health from Lifestyle Habits

# **Group 6** - Narges Zareigolbaghi & Kachina Aschenberg Øwre 
# **Kaggle names** - nargeszareigolbaghi & kachasch

## 1. Imports

- All imports in one cell at the top: numpy, pandas, matplotlib/seaborn for plots
- scikit-learn tools: Pipeline, ColumnTransformer, SimpleImputer, StandardScaler, OneHotEncoder, KFold, GridSearchCV, mean_absolute_error and the models
- One fixed `random_state` used everywhere, so results are the same every run

In [1]:
# Imports for data analysis and visualization
import numpy as np  # numerical computing
import pandas as pd  # dataframes and series
import matplotlib.pyplot as plt  # plotting
import seaborn as sns  # statistical plotting

# Imports for preprocessing, pipelines and model evaluation
from sklearn.pipeline import Pipeline  # chains preprocessing and model
from sklearn.compose import ColumnTransformer  # different preprocessing for numeric and text columns
from sklearn.impute import SimpleImputer  # fills in missing values
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.model_selection import KFold, GridSearchCV, cross_val_predict
from sklearn.metrics import mean_absolute_error  # the Kaggle metric
from sklearn.base import clone  # fresh, untrained copy of a model

# Imports for models
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# Constants
RANDOM_STATE = 42  # same random state everywhere, so results can be reproduced


## 2. Reading the data

- Read `train.csv` and `test.csv` with `id` as the index, so `id` is never used as a feature and the submission gets the right ids
- Show the shape and the first rows of both
- Split train into X (the 15 features) and y (`target`)

In [2]:
# Read the data; id becomes the index, so it is never used as a feature
df_train = pd.read_csv("train.csv", index_col="id")
df_test = pd.read_csv("test.csv", index_col="id")

# Size of the data: (rows, columns)
print("Train:", df_train.shape)
print("Test: ", df_test.shape)

# First rows of the training data
df_train.head()


Train: (367500, 16)
Test:  (150000, 15)


,age,sleep_duration_hrs,sleep_quality_score,cardio_mins_wk,water_intake_liters,screen_time_recreational_hrs,social_media_hrs,blood_pressure_sys,diet_ultra_processed_pct,diet_omega3_freq,smoking_status,alcohol_drinks_wk,novel_learning_hrs_wk,social_connection_score,chronic_stress_score,target
id,,,,,,,,,,,,,,,,
193355,55.0,NaN,9.0,20.0,0.900000,3.7,1.1,129.0,47.7,1.0,Never,3.0,1.8,8.0,8.0,27
509850,23.0,6.8,8.0,66.0,-3.634081,NaN,1.8,108.0,29.9,7.0,Never,63.0,0.9,5.0,5.0,28
413954,70.0,9.4,9.0,13.0,2.100000,9.5,5.0,137.0,NaN,2.0,Never,1.0,2.4,8.0,5.0,25
94859,40.0,7.5,10.0,200.0,1.600000,5.0,0.5,124.0,NaN,0.0,Never,7.0,0.9,NaN,8.0,29
18214,18.0,6.9,6.0,81.0,NaN,3.6,1.2,118.0,43.2,2.0,Never,0.0,0.5,5.0,10.0,26


## 3. Data exploration and visualisation (required)

- Data types, summary statistics (`describe`) and missing values per column
- Histogram of `target`: range 17-30, many people exactly at 30 (the ceiling)
- Histograms or boxplots of the features: show the impossible values (e.g. negative age, scores above 10)
- Compare value ranges in train and test (test has no missing or impossible values)
- Count duplicate rows
- Correlation heatmap of features and target (the Kaggle page recommends this)
- Score by `smoking_status` (boxplot or bar plot)
- A short comment under every plot saying what you conclude (graders check this)

## 4. Data cleaning

- Remove the exact duplicate rows
- Decide what to do with impossible values: set them to missing (NaN) or drop the rows, and say why
- Use the valid ranges (from the brief and the test set) as the limits
- Do not drop rows just because they have missing values (more than half the rows have at least one); they are filled in during preprocessing
- Do not remove or change any rows in test: all 150,000 need a prediction

## 5. Data exploration and visualisation after cleaning

- Repeat the most important plots on the cleaned data
- Correlation heatmap again: which features matter most, and which look like pure noise (e.g. `water_intake_liters`)
- Short comment: what changed after cleaning

## 6. Data preprocessing

- ColumnTransformer: numeric columns get an imputer + scaler, `smoking_status` gets an imputer + one-hot encoder
- Put all preprocessing inside the pipelines, so cross-validation never leaks information from the validation folds
- Optional feature engineering or feature selection: keep it only if CV MAE improves
- The training set is large: if grid search is too slow, tune on a random sample, then train the final model on all rows

## 7. Pipeline A - Regression (linear-regression-based, required)

- Linear-regression-based model, e.g. Ridge, Lasso or ElasticNet (optionally with polynomial features)
- GridSearchCV with KFold and MAE as the score
- Report the best hyperparameters and the CV MAE
- Compare with the simple baseline "always predict 28" (MAE 1.43)

## 8. Pipeline B - Binned classification (only if you choose B)

- Bin `target` into classes with `pd.cut` (`pd.qcut` fails with 5 or more bins because so many score 30)
- Ensemble classifier (e.g. random forest) in a pipeline, tuned with GridSearchCV
- Turn each predicted class back into a score (e.g. the median score of that bin) and compute MAE
- Try several numbers of bins (e.g. 3, 5, 10, 14) and comment on how this changes MAE (required for B)

## 9. Pipeline C - Two steps

- Step 1: ensemble classifier (e.g. random forest) that predicts score 30 vs below 30, tuned with GridSearchCV
- Step 2: regressor trained only on rows below 30, tuned with GridSearchCV
- Combine: 30 where the classifier says 30, otherwise the regressor's value
- Evaluate the combined prediction with cross-validated MAE (a manual KFold loop, since it is two pipelines)

## 10. Model comparison and best model

- Table with the CV MAE of every pipeline you built (at least two of A, B, C)
- Clearly name the best model and explain why (required)
- Check whether clipping predictions to max 30 (and rounding) lowers MAE

## 11. Kaggle submission

- Train the best model on all cleaned training data
- Predict the 150,000 test rows
- Save `submission.csv` with the columns `id,target` (same order as `example_submission.csv`)
- Write down the public leaderboard score; it must beat BEAT ME

## 12. Bonus (optional) - Fewest features

- Remove features one by one (or use sequential feature selection) and check MAE
- Find the smallest set of original features with MAE at most 10% worse than the best model
- List which features those are